# Snowflake Python API



<a id="topics"></a>
### Topics in this lesson


1. [Setup](#Setup)
1. [Snowflake API](#Snowflake_API)
1. [Create a Database](#Create_a_Database)
1. [Create a Schema](#Create_a_Schema)
1. [Create a Table](#Create_a_Table)
1. [Create a Warehouse](#Create_a_Warehouse)
1. [Create a Stage](#Create_a_Stage)
1. [Clean up](#Clean_up)  


<a id="Setup"></a>
## 1. Setup

This is a new feature so I need to install Snowflake:

In [ ]:
pip install snowflake -U

---
#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse.

*You needn't edit anything in the following cell. Just run it.*

In [ ]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

In [ ]:
# Hard code the lesson name
lesson_name = "PYTHON_API_PY"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

**For this demo we need the USERNAME:**

In [ ]:
# Retrieve the current username
from snowflake.snowpark.functions import current_user
username = (str(session.create_dataframe([""]).to_df("")
    .select(current_user())
    .collect()[0][0]
   )
)
print(f"The current user is: {username}")

# Retrieve database name
db_name = f"{username}_DB"

<a id="Snowflake_API"></a>
## 2. Snowflake API

The Python API can be used to define and manage core resources (such as tables, warehouses, and tasks) across Snowflake workloads. 

> **&#128221; Note:** Unlike the Python Connector, these APIs interact with Snowflake using native Python without the need to use SQL.


The Snowflake Python API consists of the following:



- **snowflake.core** - Defines an Iterator to represent a certain resource instances fetched from the Snowflake database
- **snowflake.core.paging**
- **snowflake.core.exceptions**
- **snowflake.core.database** - Manages Snowflake databases
- **snowflake.core.schema** - Manages Snowflake schemas
- **snowflake.core.task** - Manages Snowflake Tasks
- **snowflake.core.task.context** - Manage the context in a Snowflake Task
- **snowflake.core.task.dagv1** - A set of higher-level APIs in snowflake.core.task to more conveniently manage DAGs
- **snowflake.core.compute_pool** - Manages Snowpark Container Compute Pools
- **snowflake.core.image_repository** - Manages Snowpark Container Image Repositories
- **snowflake.core.service** - Manages Snowpark Container Services

and more 

- **snowflake.core.table** - Manages Snowflake tables
- **snowflake.core.warehouse** - Manages Virtual Warehouses 
- **snowflake.core.function** - Manages Snowflake functions 
- **snowflake.core.grant** - Manages Snowflake privileges
- **snowflake.core.role** - Manages Snowflake user roles
- **snowflake.core.user** - Manages Snowflake users
- **snowflake.core.stage** - Manages Snowflake stages
- **snowflake.core.dynamic_table** - Manages Snowflake Dynamic Tables










snowflake.core represents the entry point to the core Snowflake Python APIs that manage Snowflake objects. 

In [ ]:
from snowflake.snowpark import Session
from snowflake.core import Root
from snowflake.core.database import Database
from snowflake.core.schema import Schema
from snowflake.core.table import Table, TableColumn, PrimaryKey
from snowflake.core.warehouse import Warehouse




**Import and instantiate the Root class from snowflake.core, and pass in the Snowpark session object as an argument. You'll use the resulting Root object to use the rest of the methods and types in the Snowflake Python API.**

In [ ]:
root = Root(session)

**Let's begin with a Database.** 

The next line of code will create a database in your account called (user)_snowapi_db, and is functionally equivalent to the SQL command CREATE OR REPLACE DATABASE (user)_SNOWAPI_DB;. 

This line of code follows a common pattern for managing objects in Snowflake. 

`root.databases.create()` - is used to create a database in Snowflake. It accepts two arguments, a Database object and a mode.

We pass in a Database object with Database(name="`{session.get_current_user()}`_SNOWAPI_DB"), and set the name of the database using the name argument.

<a id="Create_a_Database"></a>
## 3. Create a Database

In this step, we will use the Snowflake Python API to create a database and then retrieve a handle to access its details.

Note the use of the specific creation mode `ifnotexists` - one of the possible options. Observe also that we confirm the creation of our object by listing databases with names containing with "snowapi" using the `iter` method.

In [ ]:
_ = session.sql("create database if not exists my_db").count()

In [ ]:
database = f"{username}_snowapi_db"

# Create the Database object
database_ref = Database(name=database)

print(database_ref)

In [ ]:
database_resource = root.databases.create(database_ref, mode="orreplace")  #ifnotexists

In [ ]:
# Get the details for the database
snowapi_database_details = database_resource.fetch()
print(snowapi_database_details.to_dict())

#### Now we check if it is indeed created:

In [ ]:
# Verify that the task_database was created successfully
database_list = root.databases.iter(like=f"{username}_snowapi%")

for databases in database_list:
  print(f"\nDatabase name: {databases.name}")

#### And set it as the current database:


In [ ]:
print(f"Setting current database in the Session instance to {database}")
session.use_database(database)

### Return the current database and schema

In [ ]:
print(session.get_current_database())
print(session.get_current_schema())

<a id="Create_a_Schema"></a>
## 4. Create a Schema

In this step, we will use the Snowflake Python API to create a schema and then retrieve a handle to access its details.


In [ ]:
# Import the Schema class
from snowflake.core.schema import Schema

schema = "snowapi_schema"

# Create the Schema object task_schema
snowapi_schema_ref = Schema(name=schema)

# Create the schema
schema_resource = root.databases[f"{username}_snowapi_db"].schemas.create(
     snowapi_schema_ref
    ,mode="orreplace"
)

# Get the details for the snowapi_schema
snowapi_schema_details = schema_resource.fetch()

print(snowapi_database_details.to_dict())

# List all the schemas in the snowapi_database
# Create a PagedIter object form the SchemaCollection object that can be looped through to list all schemas in the database
schema_list = root.databases[f"{username}_snowapi_db"].schemas.iter() 

for schema_object in schema_list:
    print(f"\nSchema name: {schema_object.name}")

In [ ]:
full_schema_name = f"{database}.{schema}"
# Set our current namespace appropriately
print(f"Setting current namespace in the Session instance to {full_schema_name}")
session.use_schema(full_schema_name)

In [ ]:
print(session.get_current_database())
print(session.get_current_schema())

<a id="Create_a_Table"></a>
## 5. Create a Table

In this step, we will use the Snowflake Python API to create a table and then retrieve a handle to access its details. 

Note also the use of the `TableColumn` class to set the column formats for the table. When used, constraints such as unique and primary keys are also configured on these column level objects.

In [ ]:
# Import the Table and TableColumn classes
from snowflake.core.table import Table, TableColumn

table = "debtor_list"


# Create the Table reference object 
# The table, debtor_list, will load the raw data
snowapi_table_ref = (Table(
    name=table,
    columns= [
         TableColumn(name="name", datatype="string", nullable=False), 
         TableColumn(name="credit_card_expiry", datatype="STRING", nullable=False), 
         TableColumn(name="credit_score", datatype="INTEGER", nullable=True)
    ]))


#Create the table
snowapi_table_resource = root.databases[database].schemas[schema].tables.create(
     snowapi_table_ref
    ,mode="ifnotexists"
)

snowapi_table_list = snowapi_table_resource.fetch().to_dict()
for key in snowapi_table_list.keys():
    print(f"{key}:\t{snowapi_table_list[key]}")

#### Let's verify if the table has been created in Snowflake
Note that the table will be empty at this point.

In [ ]:
df_debtor = session.table(table)
df_debtor.show()

<a id="Create_a_Warehouse"></a>
## 6. Create a Warehouse

In this step, we will use the Snowflake Python API to create a Snowflake virtual warehouse and then retrieve a handle to access its details. This will be configured as an `XSMALL` with an auto-suspend period of 120 seconds, to automatically pause after inactivity.

In [ ]:
# Import the Warehouse class
from snowflake.core.warehouse import Warehouse

warehouse = f"{username}_snowapi_wh"

snowapi_warehouse_ref = Warehouse(
     name=warehouse  
    ,warehouse_size="XSMALL" 
    ,auto_suspend=120       # automated timeout due to inactivity
    ,auto_resume="true"
)

# Create the warehouse 
snowapi_warehouse_resource = root.warehouses.create(snowapi_warehouse_ref,mode="orreplace")

# Retrieve and print the warehouse details
snowapi_warehouse_details = snowapi_warehouse_resource.fetch()

print(snowapi_warehouse_details.to_dict())

# List warehouses starting with "task"
warehouse_list = root.warehouses.iter(like=f"{username}_snowapi_wh")

for warehouse_object in warehouse_list:
    print(f"\nWarehouse name: {warehouse_object.name}")

#### Verify the warehouse has been created

In [ ]:
print(session.get_current_warehouse())

<a id="Create_a_Stage"></a>
## 7. Create a Stage

You can use Python to manage Snowflake stages, which are locations of data files in cloud storage.

In this step, we will use the Snowflake Python API to create a Snowflake Stage.

The Snowflake Python API represents stages with two separate types:

- `Stage`: Exposes a stage’s properties such as its name, encryption type, credentials, and directory table settings.

- `StageResource`: Exposes methods you can use to fetch a corresponding Stage object, upload and list files on the stage, and drop the stage.

In [ ]:
# Import the Stage class
from snowflake.core.stage import Stage, StageEncryption

# Create the Stage reference object 
my_stage = Stage(
  name=f"{username}_snowapi_stage",
  encryption=StageEncryption(type="SNOWFLAKE_SSE")
)

# Create the Stage
stages = root.databases[f"{username}_snowapi_db"].schemas["snowapi_schema"].stages
stages.create(my_stage)


Now that we have a Stage, we can get information about that stage by calling the StageResource.fetch method, which returns a Stage object.

In [ ]:
my_stage = root.databases[f"{username}_snowapi_db"].schemas["snowapi_schema"].stages[f"{username}_snowapi_stage"].fetch()
print(my_stage.to_dict())

In [ ]:
from snowflake.core.stage import StageCollection

# List Stages containing the word "snowapi"
stages: StageCollection = root.databases[f"{username}_snowapi_db"].schemas["snowapi_schema"].stages
stage_iter = stages.iter(like="%snowapi%")  # returns a PagedIter[Stage]

for stage_obj in stage_iter:
  print(stage_obj.name)

#### Verify the stage has been created

In [ ]:
session.sql("SHOW STAGES LIKE '%SNOWAPI%'").show()

#### Performing stage operations

You can perform common stage operations—such as uploading a file to a stage and listing files on a stage—with a StageResource object.


In [ ]:
my_stage_res = root.databases[f"{username}_snowapi_db"].schemas["snowapi_schema"].stages[f"{username}_snowapi_stage"]

# Uploads a file named my-file.yaml to the my_stage stage with the specified auto-compress and overwrite options.
my_stage_res.upload_file("diabetes.txt", "/", auto_compress=False, overwrite=True)

#Lists all files on the stage to verify that the file was uploaded successfully.
stageFiles = root.databases[f"{username}_snowapi_db"].schemas["snowapi_schema"].stages[f"{username}_snowapi_stage"].list_files()
for stageFile in stageFiles:
  print(stageFile)

In [ ]:
# We can even drop the stage again
my_stage_res.drop()

In [ ]:
# Is it gone now?
session.sql("SHOW STAGES LIKE '%SNOWAPI%'").show()

<a id="Clean_up"></a>
## 7. Clean Up

In [ ]:
# Delete database
my_db_res = root.databases[f"{username}_snowapi_db"]
my_db_res.delete()

In [ ]:
# Delete warehouse
my_wh_res = root.warehouses[f"{username}_snowapi_wh"]
my_wh_res.delete()

In [ ]:
close_session_and_clean_up(get_lesson())